                # GEPA

                Use textual failure feedback to evolve instructions, preserving reflection logs and best outputs along the way.

                **Use it when:** Your metric can explain errors, not merely score them, and you want a detailed instruction that encodes those lessons.

                **What compilation changes:** In this rerun GEPA learned a long standalone instruction and no final demonstrations.

                Important configuration in this benchmark:

                - six full evaluations in the full profile
- reflection minibatches of three
- feedback metric returns both score and diagnosis
- seed 42; merge disabled for a bounded run

                Every notebook loads the same 74-row dataset and frozen, pair-grouped
                train/validation/test membership before it can compile anything.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import artifact_paths, learned_program_preview, verify_prompt_artifact
from chapter06.optimizer_runtime import (
    format_result,
    load_frozen_examples,
    published_result,
    run_optimizer,
    split_summary,
)

OPTIMIZER = 'gepa'
splits = load_frozen_examples()
RUN_LIVE = os.getenv("CHAPTER06_RUN_LIVE", "0") == "1"
print(f"optimizer={OPTIMIZER!r}; live={RUN_LIVE}")
print(split_summary(splits))

optimizer='gepa'; live=False
train=36 (human=18, AI=18); validation=18 (human=9, AI=9); test=20 (human=10, AI=10)


                ## Compile shape

                This is the essential DSPy call used by the shared executable runner:

                ```python
                optimizer = dspy.GEPA(
    metric=feedback_metric, max_full_evals=profile.gepa_max_full_evals,
    reflection_minibatch_size=3, reflection_lm=reflection_lm,
    use_merge=False, track_best_outputs=True, seed=42,
)
optimized_detector = optimizer.compile(detector, trainset=trainset, valset=valset)
                ```

                `compile` returns a program. The shared runner then evaluates that program on the
                untouched 20-example test split. The baseline has its own notebook; all other
                notebooks report the optimized program's final test accuracy directly.

In [2]:
if RUN_LIVE:
    live_run = run_optimizer(
        OPTIMIZER,
        splits=splits,
    )
    result = live_run.summary()
else:
    result = published_result(OPTIMIZER)

print(format_result(result))
print()
print(artifact_paths(OPTIMIZER))

optimizer: gepa
task model: openai/gpt-5.6-luna
final test accuracy: 90.0% (18/20)
optimization time: 900.7s

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/gepa.json
- canonical prompt: chapter06/results/final_prompts/gepa.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

The preview is deliberately truncated; follow the prompt path for the complete learned rule set and `optimizer_logs/` for the evolutionary trace.

The saved output above uses the checked-in result so opening or running the notebook
is cheap. Set `CHAPTER06_RUN_LIVE=1` before launching Jupyter to execute the real
optimizer; prompt optimizers require an OpenAI key, while weight optimizers also
require the local PyTorch/Transformers stack. The next cell previews the published
program artifact.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (9745 characters):
You are a provenance classifier. Your task is to determine whether a passage was most likely generated or rewritten by AI.

You will receive exactly one input field:

### text
The passage to classify.

Return exactly these two fields and no other content:

### reasoning
A concise explanation citing concrete, passage-specific evidence from diction, phrasing, rhythm, specificity, organization, transitions, or natural irregularities.

### is_ai
True or False

Use `True` when the passage was most likely AI-generated or AI-rewritten. Use `False` when it was most likely human-authored.

Evaluate the passage holistically. Do not decide from a single word, phrase, technical term, or superficial feature. Grammatical polish, formality, repetition, promotional language, parenthetical definitions, instructional structure, and technical sophistication are not sufficient by themselves. Conversely, concrete technical details do not auto

## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Keep the test set untouched until the optimizer returns,
then report final accuracy as `correct / test examples` so every optimizer is easy
to compare. Use the separate baseline notebook when you also need uplift.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.